# LIBERO 재학습+재eval — **노드 5 전용** (노드 5대 × GPU 2, seed2 우선)

**기존 방식 그대로**: 학습=`run_training_jobs`(resume/skip), eval=`run_libero_eval_jobs`.
이 노드 몫만, **2-GPU 청크마다 셀**. 학습 셀들 먼저 → eval 셀들.

- 유효 eval = overall n_ep ≥ 2500(=500×10) + action + ckpt ≥150k. 옛 50ep·미완·under 는 무효.
- under-trained(70k 등)는 **지우지 않고 resume** 으로 150k 까지 이어학습. eval 은 150k 체크포인트.
- ① 은 **무효 eval_info(파일)만 삭제**(폴더 rmtree 안 함 → NFS 잠금 방지). 그 seed 안 돌 때 실행.
- ⚠️ eval 청크당 ~40시간. 안 죽는 환경(nohup/tmux).


In [ ]:
import sys, json
from pathlib import Path
_here = Path.cwd()
_root = next(c for c in (_here, *_here.parents)
             if (c / 'common_final.py').exists() or (c / 'notebooks' / 'common_final.py').exists())
_root = _root / 'notebooks' if (_root / 'notebooks' / 'common_final.py').exists() else _root
sys.path.insert(0, str(_root))
import importlib, common_final as cf
importlib.reload(cf)

TASK   = 'libero_10'
TARGET = cf.CKPT_STEP                     # 150,000
MIN_VALID_EP = 10 * cf.EVAL_N_EP // 2     # 유효 500ep = overall n_ep >= 2500(=5000). 옛 50ep(500) 제외
MODELS  = ['act', 'acm', 'acm2', 'mosaic', 'bimamba', 'bimamba_s7']
TAG_CFG = {'act': 'act', 'acm': 'acm', 'acm2': 'acm2', 'bimamba': 'bimamba',
           'bimamba_s7': 'bimamba_mosaic', 'mosaic': 'mosaic_infer'}
for folder, cfgkey in TAG_CFG.items():
    cf.v23.MODEL_DIR_NAMES.setdefault(folder, folder)
    if folder not in cf.v23.MODEL_CONFIGS:
        cf.v23.MODEL_CONFIGS[folder] = cf.v23.MODEL_CONFIGS[cfgkey]

GPU_COUNTS = [2, 2, 2, 2, 2]   # 클러스터: 노드 5대 × GPU 2
NODE_IDX   = 4               # 이 노트북 = 노드 5 전용
TRAIN_ROOT = cf.OUTPUT_BASE / 'train' / TASK
EVAL_ROOT  = cf.OUTPUT_BASE / 'eval_clean' / TASK
print('노드', NODE_IDX, '| 목표', f'{TARGET:,}', '| 유효 eval overall n_ep >=', MIN_VALID_EP)

## 상태 — 이 노드 학습/eval 청크


In [ ]:
# ── 상태 분류 → 이 노드 몫 (seed2 우선 라운드로빈). 유닛=(모델,seed) ──
def tstep(t, s):
    return cf.v23.last_ckpt_step(TRAIN_ROOT / t / f'seed{s}')

def eval_rec(t, s):
    d = EVAL_ROOT / t / f'seed{s}'
    if not d.is_dir():
        return None
    best = None
    for info in d.rglob('eval_info.json'):
        try:
            ov = json.loads(info.read_text()).get('overall', {})
        except Exception:
            continue
        ne = ov.get('n_ep', ov.get('n_episodes')) or 0
        if best is None or ne > best['n_ep']:
            best = {'n_ep': ne, 'has_act': (info.parent / 'actions').is_dir(), 'info': info}
    return best

SEED_ORDER = [2, 1, 3, 0]        # seed2 먼저 (baseline 3개 이미 done → 신규 모델만 채우면 완성)
N_NODES = len(GPU_COUNTS)
units = []                       # (t, s, step, needs_train, ev, valid)
for s in SEED_ORDER:
    for t in MODELS:
        step = tstep(t, s)
        nt = step is None or step < TARGET
        ev = eval_rec(t, s)
        ve = bool(ev and ev['n_ep'] >= MIN_VALID_EP and ev['has_act'] and not nt)
        if nt or not ve:
            units.append((t, s, step, nt, ev, ve))

mine = units[NODE_IDX::N_NODES]
mine_train = [(t, s, TASK) for (t, s, step, nt, ev, ve) in mine if nt]
mine_eval  = [(t, s) for (t, s, step, nt, ev, ve) in mine]
# 재학습(또는 무효) 대상이 갖고 있는 '무효 eval_info' → 재eval 이 skip 되지 않게 파일만 지운다(폴더 rmtree X)
stale_infos = [ev['info'] for (t, s, step, nt, ev, ve) in mine if ev and not ve]

NG = GPU_COUNTS[NODE_IDX]
train_chunks = [mine_train[i:i + NG] for i in range(0, len(mine_train), NG)]
eval_chunks  = [mine_eval[i:i + NG] for i in range(0, len(mine_eval), NG)]
print(f'이 노드: 학습 {len(mine_train)}({len(train_chunks)}청크) / eval {len(mine_eval)}({len(eval_chunks)}청크)')
for k, c in enumerate(train_chunks):
    print(f'   학습 청크 {k + 1}: {[(t, s) for t, s, _ in c]}')
for k, c in enumerate(eval_chunks):
    print(f'   eval  청크 {k + 1}: {c}')
print(f'무효 eval_info(재eval 위해 ①에서 삭제): {len(stale_infos)}개')

## ① 무효 eval_info 삭제 (재eval 위해, 파일만) — dry-run → EXECUTE=True
그 seed 의 학습/eval 이 **안 돌고 있을 때** 실행. under-trained 체크포인트는 안 건드림(②에서 resume).


In [ ]:
# ── ① 무효 eval_info.json 삭제 (재eval 이 skip 안 되게) ── 파일만, 폴더 rmtree 안 함 ──
#    under-trained 체크포인트는 안 지운다 → ②에서 resume 으로 150k 까지 이어학습(기존 방식).
#    ⚠️ 이 셀은 그 seed 의 학습/eval 이 안 돌고 있을 때 실행할 것.
EXECUTE = False

if not stale_infos:
    print('삭제할 무효 eval_info 없음.')
else:
    print(f'{"삭제" if EXECUTE else "DRY-RUN"} — 무효 eval_info {len(stale_infos)}개:')
    for inf in stale_infos:
        print(f'   {inf.relative_to(EVAL_ROOT)}')
        if EXECUTE and inf.exists():
            inf.unlink()
    print('\n' + ('삭제 완료.' if EXECUTE else '확인됐으면 EXECUTE=True.'))

## ② 학습 — 2-GPU 청크마다 셀 (목표 150k)


In [ ]:
# ── ② 학습 청크 1 (GPU 2개) — 목표 150k (resume/skip 자동) ──
K = 0
gpus = cf.available_gpus()[:GPU_COUNTS[NODE_IDX]]
C = train_chunks[K] if K < len(train_chunks) else []
if C:
    cf.run_training_jobs(C, gpus=gpus, prefetch_task=TASK)
else:
    print(f'학습 청크 {K + 1} 없음')

In [ ]:
# ── ② 학습 청크 2 (GPU 2개) — 목표 150k (resume/skip 자동) ──
K = 1
gpus = cf.available_gpus()[:GPU_COUNTS[NODE_IDX]]
C = train_chunks[K] if K < len(train_chunks) else []
if C:
    cf.run_training_jobs(C, gpus=gpus, prefetch_task=TASK)
else:
    print(f'학습 청크 {K + 1} 없음')

In [ ]:
# ── ② 학습 청크 3 (GPU 2개) — 목표 150k (resume/skip 자동) ──
K = 2
gpus = cf.available_gpus()[:GPU_COUNTS[NODE_IDX]]
C = train_chunks[K] if K < len(train_chunks) else []
if C:
    cf.run_training_jobs(C, gpus=gpus, prefetch_task=TASK)
else:
    print(f'학습 청크 {K + 1} 없음')

## ③ eval — 2-GPU 청크마다 셀 (500ep). ② 다 끝난 뒤. ⚠️ 청크당 ~40시간


In [ ]:
# ── ③ eval 청크 1 (GPU 2개, 500ep) — ② 학습 다 끝난 뒤 ── ⚠️ ~40시간 ──
K = 0
gpus = cf.available_gpus()[:GPU_COUNTS[NODE_IDX]]
C = eval_chunks[K] if K < len(eval_chunks) else []
if C:
    cf.run_libero_eval_jobs(C, gpus=gpus, n_episodes=cf.EVAL_N_EP)
else:
    print(f'eval 청크 {K + 1} 없음')

In [ ]:
# ── ③ eval 청크 2 (GPU 2개, 500ep) — ② 학습 다 끝난 뒤 ── ⚠️ ~40시간 ──
K = 1
gpus = cf.available_gpus()[:GPU_COUNTS[NODE_IDX]]
C = eval_chunks[K] if K < len(eval_chunks) else []
if C:
    cf.run_libero_eval_jobs(C, gpus=gpus, n_episodes=cf.EVAL_N_EP)
else:
    print(f'eval 청크 {K + 1} 없음')

In [ ]:
# ── ③ eval 청크 3 (GPU 2개, 500ep) — ② 학습 다 끝난 뒤 ── ⚠️ ~40시간 ──
K = 2
gpus = cf.available_gpus()[:GPU_COUNTS[NODE_IDX]]
C = eval_chunks[K] if K < len(eval_chunks) else []
if C:
    cf.run_libero_eval_jobs(C, gpus=gpus, n_episodes=cf.EVAL_N_EP)
else:
    print(f'eval 청크 {K + 1} 없음')